[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/basic_kernels.ipynb)



# Basic kernels and compounds with `gpc`



GPc twin of [`GPy/basic_kernels.ipynb`](../GPy/basic_kernels.ipynb) (Nicolas Durrande / James Hensman): construct kernels, combine them, evaluate with `k.K` / thin `k.plot`, and attach them to regression.



`gpc.kern.*` objects are **descriptors** for the C++ core (CIP-0038). They print and compose with `+` / `*`, and (from `gpc>=0.1.0a10`) expose GPy-shaped `k.K(X)` and `k.plot(...)`.



**CIP-0027.** Observation noise lives on `likelihood.variance`. `Bias` / `White` are opt-in kernel terms — do not confuse kernel `White` with Gaussian observation noise.



```python

k = gpc.kern.RBF(1) + gpc.kern.Bias(1)

K = k.K(X)

k.plot()

m = gpc.models.GPRegression(X, Y, kernel=k)

```



In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d


## Basics: construction and printing

For most kernels the input dimension is the only mandatory argument; variance / lengthscale default to 1.


In [ ]:
ker1 = gpc.kern.RBF(1)  # variance=1, lengthscale=1
ker2 = gpc.kern.RBF(1, variance=0.5, lengthscale=0.2)
ker3 = gpc.kern.RBF(1, variance=1.0, lengthscale=2.0)
print(ker1)
print(ker2)
print(ker3)
print("parts:", ker2.to_parts())


## Catalogue (CIP-0038)

A short tour of kernels available on the façade. Construct and print — the GPy notebook plots \(k(x,0)\); here we keep the checklist as an inventory.


In [ ]:
kerns = [
    gpc.kern.RBF(1),
    gpc.kern.ExpQuad(1),  # same family; GPy ExpQuad path names (CIP-0024)
    gpc.kern.Exponential(1),
    gpc.kern.Matern32(1),
    gpc.kern.Matern52(1),
    gpc.kern.Brownian(1),
    gpc.kern.Linear(1),
    gpc.kern.Bias(1),
    gpc.kern.White(1),
    gpc.kern.StdPeriodic(1, period=1.0, lengthscale=0.5),
    gpc.kern.Cosine(1),
    gpc.kern.RatQuad(1),
    gpc.kern.MLP(1),
]
for k in kerns:
    print(f"{type(k).__name__:14s} -> {k}")


## Covariance matrix and thin kernel plot



GPy's `basic_kernels` plots \(k(x,0)\). On the façade the same geometry is available without fitting a model: `k.K(X)` returns the Gram matrix; `k.plot` draws a 1-D slice (requires matplotlib / `gpc[plot]`).


In [ ]:
Xgrid = np.linspace(-1.0, 1.0, 80)[:, None]

k_show = gpc.kern.RBF(1, lengthscale=0.25, variance=1.0) + gpc.kern.Bias(1, variance=0.25)

print("K(X) shape:", k_show.K(Xgrid).shape)

print("K diag sample:", np.diag(k_show.K(Xgrid))[:5])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

# Slice k(x, 0) for a few catalogue kernels (GPy basic_kernels style).

x0 = np.zeros((1, 1))

for name, k in [

    ("RBF", gpc.kern.RBF(1, lengthscale=0.3)),

    ("Matern32", gpc.kern.Matern32(1, lengthscale=0.3)),

    ("Linear", gpc.kern.Linear(1)),

]:

    axes[0].plot(Xgrid[:, 0], k.K(Xgrid, x0)[:, 0], label=name)

axes[0].set_title(r"$k(x, 0)$")

axes[0].set_xlabel("$x$")

axes[0].legend(loc="best", fontsize=8)

k_show.plot(ax=axes[1], plot_limits=(-1.0, 1.0))

axes[1].set_title("k.plot (RBF + Bias)")

fig.tight_layout()

plt.show()


## Combining kernels

Add or multiply descriptors to build compounds. Parameters are snapshotted into the compound at construction (unlike GPy, where child objects often remain live-linked).

Unlike GPy, editing `k1` after `k_add = k1 + k2` does **not** mutate the compound — parameters were snapshotted at construction.


In [ ]:
k1 = gpc.kern.RBF(1, variance=1.0, lengthscale=2.0)
k2 = gpc.kern.Matern32(1, variance=0.5, lengthscale=0.2)
k_prod = k1 * k2
k_add = k1 + k2
print("product:", k_prod)
print("sum:    ", k_add)
print("sum parts:", k_add.to_parts())
print("prod parts:", k_prod.to_parts())


## Compound on a 1-D regression + paths

Attach a sum kernel to `GPRegression` and set hyperparameters by path. Compare with a signal-only string kernel (`kernel='rbf'`), which leaves mean structure to the likelihood / optional Bias.


In [ ]:
rng = np.random.default_rng(1)
X = np.linspace(0.0, 1.0, 30)[:, None]
Y = 0.4 + np.sin(2 * np.pi * X) + 0.05 * rng.normal(size=X.shape)

k = gpc.kern.RBF(1) + gpc.kern.Bias(1)
m = gpc.models.GPRegression(X, Y, kernel=k)
m.set_by_path("rbf.lengthscale", 0.25)
m.set_by_path("rbf.variance", 1.0)
m.set_by_path("bias.variance", 0.25)
m.set_by_path("likelihood.variance", 0.05)
print("paths:", m.param_paths())
print(m)
m.optimize(iters=60, messages=False)
print("objective:", m.objective())
print(m)

m_sig = gpc.GPRegression(X, Y, kernel="rbf")
m_sig.set_by_path("rbf.lengthscale", 0.25)
m_sig.optimize(iters=60, messages=False)

Xtest = np.linspace(-0.1, 1.1, 200)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[0], label="RBF+Bias")
axes[0].set_title("kern.RBF + kern.Bias")
plot_1d(m_sig, X, Y, Xtest=Xtest, ax=axes[1], label="string rbf")
axes[1].set_title("kernel='rbf' (signal only)")
fig.tight_layout()
plt.show()


## Operating on different domains (`active_dims`)

Factors in a sum can act on different columns of \(X\). Here a linear trend on dimension 0 plus an RBF on dimension 1.


In [ ]:
rng = np.random.default_rng(0)
X2 = rng.normal(size=(40, 2))
Y2 = np.sin(X2[:, :1]) + 0.3 * X2[:, 1:] + 0.05 * rng.normal(size=(40, 1))

k2 = gpc.kern.Linear(1, active_dims=[0]) + gpc.kern.RBF(1, lengthscale=1.0, active_dims=[1])
m2 = gpc.models.GPRegression(X2, Y2, kernel=k2)
print(m2)
print("paths:", m2.param_paths())
m2.optimize(iters=50, messages=False)
print("objective:", m2.objective())
print(
    "linear.variance",
    m2.get_by_path("linear.variance"),
    "rbf.lengthscale",
    m2.get_by_path("rbf.lengthscale"),
)


## Product vs sum on the 1-D toy

Same two factors as above; sum is usually the right default for “trend + local”, while a product modulates one covariance by another.


In [ ]:
k_sum = gpc.kern.RBF(1, lengthscale=0.3) + gpc.kern.Linear(1)
k_mul = gpc.kern.RBF(1, lengthscale=0.3) * gpc.kern.Linear(1)
m_sum = gpc.models.GPRegression(X, Y, kernel=k_sum)
m_mul = gpc.models.GPRegression(X, Y, kernel=k_mul)
m_sum.optimize(iters=50, messages=False)
m_mul.optimize(iters=50, messages=False)
print("sum objective", m_sum.objective())
print("prod objective", m_mul.objective())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m_sum, X, Y, Xtest=Xtest, ax=axes[0], label="RBF+Linear")
axes[0].set_title("sum")
plot_1d(m_mul, X, Y, Xtest=Xtest, ax=axes[1], label="RBF*Linear")
axes[1].set_title("product")
fig.tight_layout()
plt.show()
